# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abc085455-byte/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Numeric fields are used mostly as-is, with three exceptions: the four volume fields
(`impressions_90d`, `clicks_90d`, `sessions_90d`, `ai_sessions_90d`) get `log1p()` because they're
heavily right-skewed (Section 1 of `w04_signal_audit.ipynb` shows a ~700x median-to-max spread on
`impressions_90d`), and `search_volume` / `word_count` get an explicit missingness flag instead of
a blind zero-fill, because the data dictionary notes both go missing in a pattern tied to
`content_type` rather than at random — filling with 0 would silently tell the model "no search
demand" when the truth is "not measured for this content type."


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "has_search_volume", "has_word_count",
]
categorical_features = [
    "content_type", "main_intent", "competition_level", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_search_volume"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[categorical_features].fillna("unknown").astype(str)
cat_enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_enc.reset_index(drop=True)], axis=1)

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns "
      f"({len(numeric_features)} numeric + {X.shape[1] - len(numeric_features)} one-hot encoded)")
print(f"Any remaining NaNs: {X.isna().any().any()}")


Feature matrix: 30,000 rows x 53 columns (19 numeric + 34 one-hot encoded)
Any remaining NaNs: False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `log_impressions_90d`/`clicks`/`sessions`/`ai_sessions` | log1p of trailing-90d volume | none expected; would `fillna(0)` if any | Same 90-day window as the label — expected overlap for this same-period triage task (see `w06_validation_audit.ipynb` Section 3). |
| `days_with_impressions` | count of the last 90 days with >=1 impression | none expected | Same window as above. |
| `content_age_days` | days since first publish | none expected | Known at any time — a pure content-metadata field. |
| `days_since_last_update` | days since last edit | none expected | Known at any time. |
| `ctr`, `avg_position` | trailing-90d rate / average rank | `avg_position == 0` means "no position data," not rank zero — kept as-is, not imputed, since it's a real, meaningful category (very low visibility). | Same 90-day window. |
| `search_volume`, `competition`, `cpc` | keyword-difficulty fields | `has_search_volume` flag added; raw field filled with 0 only where the flag already marks it missing | Independent of the label window — external keyword metadata. |
| `word_count` | page length | `has_word_count` flag added, same reasoning | Known at any time. |
| `content_type`, `main_intent`, `competition_level`, `*_tier` | categorical buckets | filled with the literal string `"unknown"` before one-hot encoding, so missingness becomes its own visible category | Known at any time / same window, depending on field. |


In [2]:
print("Missing-value audit (raw columns before fill):")
for col in ["search_volume", "word_count", "avg_position", "ctr"]:
    n_missing = df[col].isna().sum()
    print(f"  {col:16s} missing: {n_missing:5d} ({n_missing/len(df):.1%})")

print("\navg_position == 0 count (no position data, not rank zero):", (df["avg_position"] == 0).sum())


Missing-value audit (raw columns before fill):
  search_volume    missing:  2468 (8.2%)
  word_count       missing:  7699 (25.7%)
  avg_position     missing:     0 (0.0%)
  ctr              missing:     0 (0.0%)

avg_position == 0 count (no position data, not rank zero): 1205


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

**Label-derived columns.** `trend_direction` (the label itself) and `trend_pct` (its numerator)
are never in the feature set. Confession test: deliberately adding `trend_pct` back in lifts a
Logistic Regression's ROC AUC on a held-out split from ~0.61 to ~0.999 — confirming both that the
test harness is sensitive enough to catch a real leak, and that this column must stay excluded.

**Future windows.** This dataset is a single trailing-90-day snapshot, not a repeated time series
per page, so there's no literal "future" row to leak from. The closer risk is *within-window*
overlap: `impressions_90d` (and everything summed over it) technically contains the label's own
`last_30d` / `prev_30d` sub-windows. That's expected for a same-period triage task ("is this page
*currently* declining") rather than a forecast, and the correlation between `log1p(impressions_90d)`
and the label is weak (0.18) — nowhere near what a disguised copy of the label would show.

**Product flags / IDs.** A column-name scan for `flag`, `score`, `health`, `optimiz` finds nothing
in this starter file — there are no FlyRank product flags to accidentally reuse. `content_id` and
`client_id` are pseudonyms; they're excluded from the feature matrix entirely and used only to
group the train/test split.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

y = df["is_declining_label"].to_numpy()
groups = df["client_id"].to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

def auc_with(extra_cols):
    Xt, Xte = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
    for c in extra_cols:
        Xt[c] = pd.to_numeric(df.iloc[train_idx][c], errors="coerce").fillna(0).to_numpy()
        Xte[c] = pd.to_numeric(df.iloc[test_idx][c], errors="coerce").fillna(0).to_numpy()
    pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    pipe.fit(Xt, y[train_idx])
    return roc_auc_score(y[test_idx], pipe.predict_proba(Xte)[:, 1])

honest_auc = auc_with([])
leak_auc = auc_with(["trend_pct"])
print(f"Honest feature set ROC AUC: {honest_auc:.3f}")
print(f"+ trend_pct injected (should jump toward 1.0): {leak_auc:.3f}")

corr = np.corrcoef(np.log1p(df["impressions_90d"]), y)[0, 1]
print(f"\ncorr(log1p(impressions_90d), label): {corr:.3f}  (weak -- not a disguised copy)")

flagged_cols = [c for c in df.columns if any(k in c.lower() for k in ["flag", "score", "health", "optimiz"])]
print(f"\nColumns that look like product flags/scores: {flagged_cols if flagged_cols else 'none found'}")


Honest feature set ROC AUC: 0.615
+ trend_pct injected (should jump toward 1.0): 0.999

corr(log1p(impressions_90d), label): 0.177  (weak -- not a disguised copy)

Columns that look like product flags/scores: none found


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `trend_direction` | This **is** the label — using it as a feature would be circular. |
| `trend_pct` | The label's own numerator — confirmed leaky by the confession test above (AUC -> ~1.0). |
| `content_id` | Pseudonym, unique per row — no predictive meaning, used nowhere. |
| `client_id` | Pseudonym for grouping only — used exclusively in the train/test split, never as a feature. |
| `provider_used` / `model_used` (if present in a given release) | Data-dictionary note: describes how the row was generated, not a property of the content — not a real-world signal available at prediction time. |


In [4]:
excluded = ["trend_direction", "trend_pct", "content_id", "client_id"]
present = [c for c in excluded if c in df.columns]
in_features = [c for c in present if c in X.columns]
print("Excluded columns present in raw data:", present)
print("Any of them leaked into the feature matrix X?", bool(in_features), in_features)


Excluded columns present in raw data: ['trend_direction', 'trend_pct', 'content_id', 'client_id']
Any of them leaked into the feature matrix X? False []


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
